# SCI1070 — Semaine 7 : mesurer avant de changer, tester avant d'entraîner

| Volet | Ce que vous y faites |
| --- | --- |
| A | profiler et accélérer votre socle, **avec graines et intervalles** |
| B | écrire l'environnement de négociation, et vérifier sa récompense par des assertions |

Le **volet A est la matière du jalon 2**, remis à la fin de cette même semaine.

> **Un avertissement sur les chiffres.** Je ne vous donne, dans aucun volet, le débit
> ou la durée que vous devez obtenir. Les 229 trames par seconde, les 52 minutes, les
> +72,5 % viennent d'une machine précise sur un jeu précis. Ce que j'attends est
> **relatif** : des rapports, un classement, un signe, et un intervalle autour de
> chacun. **Si vos rapports sont stables sur vos graines, vous avez un résultat.**

In [ ]:
import importlib.util
import subprocess
import sys

for paquet, module in [("gymnasium[classic-control]>=1.0", "gymnasium"), ("torch", "torch")]:
    if importlib.util.find_spec(module) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", paquet], check=True)

# Le code fourni du cours. À partir de cette semaine, le protocole de mesure ne se
# recopie plus d'un carnet à l'autre : il est dans le paquet, et il est testé.
if importlib.util.find_spec("sci1070") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
                    "git+https://github.com/FaycalZM/drl-course-teluq.git#subdirectory=code"],
                   check=True)

import collections
import time

import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from sci1070.protocole import mesurer, intervalle, se_recouvrent
from sci1070.graines import fixer_graines

print("sci1070 chargé")

---
# Volet A : mesurer, puis changer une seule chose

### Étape 0, et elle précède toute exécution

Écrivez, **avant de lancer quoi que ce soit**, le **critère d'arrêt**, les **deux
indicateurs**, et la **liste de vos graines**.

**Un critère d'arrêt choisi après avoir vu les courbes n'est plus un critère, c'est
une justification.**

In [ ]:
CRITERE_ARRET = 475          # moyenne sur les 100 derniers épisodes, sur CartPole-v1
INDICATEURS = ("trames par seconde", "temps d'horloge jusqu'au seuil")
GRAINES = [0, 1, 2, 3, 4]    # k >= 3, 5 de préférence

print(f"critère : {CRITERE_ARRET} | indicateurs : {INDICATEURS} | graines : {GRAINES}")

### Étape 1, la version de référence

In [ ]:
class ReseauQ(nn.Module):
    def __init__(self, n_obs, n_actions, n_cache=128):
        super().__init__()
        self.pipe = nn.Sequential(
            nn.Linear(n_obs, n_cache), nn.ReLU(),
            nn.Linear(n_cache, n_cache), nn.ReLU(),
            nn.Linear(n_cache, n_actions),
        )

    def forward(self, x):
        return self.pipe(x)


Transition = collections.namedtuple(
    "Transition", ["obs", "action", "recompense", "termine", "obs_suivante"])


def une_execution(configuration, graine, seuil):
    """Rend (trames_par_seconde, temps_jusqu_au_seuil). UNE exécution, UNE graine."""
    ...        # À COMPLÉTER : votre socle de la semaine 6, instrumenté. Deux compteurs : les trames consommées, et le temps d'horloge. Le compteur compte les PAS DE L'AGENT.


debits_ref, temps_ref = mesurer({}, GRAINES, seuil=CRITERE_ARRET,
                                execution=une_execution)

for nom, x in [("débit (trames/s)", debits_ref), ("temps (s)", temps_ref)]:
    moyenne, bas, haut = intervalle(x)
    print(f"{nom:22s} {moyenne:8.1f}   [{bas:.1f}, {haut:.1f}]   n={len(x)}")

### Étape 2, une modification à la fois

In [ ]:
def profiler_par_couche(env_id="CartPole-v1", pas=1000):
    """Rend un dictionnaire {couche: secondes} sur `pas` pas d'agent."""
    ...        # À COMPLÉTER : quatre compteurs, time.perf_counter()


for couche, secondes in sorted(profiler_par_couche().items(), key=lambda kv: -kv[1]):
    print(f"{couche:28s} {secondes:7.3f} s")

In [ ]:
variantes = {
    "référence":            {},
    "sans no_grad":         {"sans_no_grad": True},
    "N=2 vectorisé":        {"n_envs": 2},
    "N=4 vectorisé":        {"n_envs": 4},
    "tenseurs préalloués":  {"tenseurs_prealloues": True},
}

...        # À COMPLÉTER : mesurer chaque variante sur LES MÊMES graines, puis une ligne par variante avec moyenne, intervalle et rapport à la référence

### La courbe en fonction de \(N\)

Deux axes, parce que les deux indicateurs **ne vont pas dans le même sens**.

In [ ]:
Ns = [1, 2, 3, 4]
...        # À COMPLÉTER : débit à gauche, temps à droite, barres d'intervalle sur les deux
ax.set_xlabel("N environnements vectorisés")
ax.set_xticks(Ns)
plt.show()

### Étape 3, le tableau et sa lecture

Avant d'annoncer un gain, posez-vous la question que `se_recouvrent` répond.

In [ ]:
# La règle d'interprétation de la section 7.2, en une ligne.
a = [228.0, 227.0, 229.0, 228.5, 227.5]      # la référence, cinq graines
b = [231.0, 230.0, 232.0, 229.5, 231.5]      # une variante « 1,5 % plus rapide »
print("les intervalles se recouvrent :", se_recouvrent(a, b))
print("-> un écart dont les intervalles se recouvrent n'est PAS un résultat")

c = [300.0, 298.0, 305.0, 301.0, 299.0]      # une variante franchement plus rapide
print("\nréférence contre +32 % :", se_recouvrent(a, c))

**Attendu du volet A.** Le tableau, les deux courbes en fonction de \(N\), vos
intervalles, **votre matériel**, et un paragraphe de lecture répondant à **une**
question : avez-vous observé un cas où le classement en débit et le classement en
temps se **contredisent** ?

---
# Volet B : écrire l'environnement, et le tester sans l'entraîner

**Le jeu de données est fourni**, vous n'avez aucune donnée de marché à aller
chercher.

In [ ]:
from sci1070.donnees import charger_prix, series_disponibles

print("séries fournies :", series_disponibles())

prix = charger_prix("synthetique-a")
print(f"{len(prix)} barres | clôture de {prix.cloture.min():.2f} à {prix.cloture.max():.2f}")

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.plot(prix.cloture, linewidth=0.8)
ax.set_xlabel("barre"); ax.set_ylabel("clôture")
ax.set_title("synthetique-a : la série sur laquelle vous travaillerez")
ax.grid(alpha=0.3)
plt.show()

Deux séries sont là pour vos assertions plutôt que pour un marché : **`plate`**, où
rien ne bouge, et **`escalier`**, une hausse de 10 % à chaque barre.

In [ ]:
plate = charger_prix("plate")
escalier = charger_prix("escalier")
print("plate     : clôtures", plate.cloture[:4], "...")
print("escalier  : clôtures", np.round(escalier.cloture[:4], 3), "...")
print("escalier  : +%.0f %% par barre" % (100 * (escalier.cloture[1] / escalier.cloture[0] - 1)))

In [ ]:
...        # À COMPLÉTER : la classe d'état et l'habillage Gym de la section 7.10. b = 10 barres, kappa = 0,1 %, trois actions, observation à 3b + 2 composantes. La POSITION dans l'état est ce qui fait de ce problème un processus de décision markovien.

### Les quatre assertions, et elles sont le cœur du volet

In [ ]:
...        # À COMPLÉTER : 1. LE COÛT DU VOYAGE. Un achat suivi d'une clôture, SANS mouvement de prix, rend exactement -2*kappa.

In [ ]:
...        # À COMPLÉTER : 2. LE RENDEMENT PUR. Avec kappa = 0, un achat en c0 et une clôture en c1 rendent exactement 100 * (c1/c0 - 1).

In [ ]:
...        # À COMPLÉTER : 3. LES DEUX SCHÉMAS NE COÏNCIDENT PAS. Ils s'accordent au premier ordre et diffèrent au second. Le cas à deux barres à +10 % donne 20 contre 21.

In [ ]:
...        # À COMPLÉTER : 4. LA FORME DE L'OBSERVATION. Exactement 3b + 2 composantes, et la DERNIÈRE vaut 0 quand aucune position n'est détenue. Sur plusieurs b.

### Deux extensions facultatives, et je les recommande toutes les deux

In [ ]:
...        # À COMPLÉTER : l'affichage : la série de l'épisode, avec les achats et les clôtures marqués ; et la stratégie d'achat-conservation

> **Mon message pour ce laboratoire.** Les deux volets vous enseignent le même geste
> par deux chemins opposés. Au volet A, vous **mesurez avant de changer**. Au volet
> B, vous **testez avant d'entraîner**. Dans les deux cas, vous refusez de lancer la
> grosse machine avant d'avoir établi un point de référence vérifiable.